In [98]:
import pandas as pd
import numpy as np
import sklearn

In [99]:
from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV

In [100]:
url = r"https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
df = pd.read_csv(url, sep=',')
df.head(5)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [101]:
df.info()
df.shape

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


(768, 9)

In [102]:
#Detectar los valores ceros y reemplazarlos por NaN
df_Nan = df.replace(0, np.nan)
count_Nan = df_Nan.drop(columns="Outcome").isnull().sum().sort_values(ascending=False)
Nan_distribution = pd.DataFrame(count_Nan, columns=['Count_Nan'])
Nan_distribution['Percentage'] = ((Nan_distribution['Count_Nan'] / len(df_Nan)) * 100).round(2) 

In [103]:
#Medicion de correlacion
df_corr = df.corr()["Outcome"].drop("Outcome").sort_values(ascending=False)
#Medicion de Skew
df_skew = df.skew()

In [104]:
Nan_distribution["Correlation"]=df_corr
Nan_distribution["Skew"]=df_skew

In [105]:
#Normalizando datos de acuerdo con la simetria
def Scaler (Skew):
    if -.5 < Skew <0.5:
        return 'StandardScaler'
    elif -1 < Skew <= -0.5 or 0.5 <= Skew < 1:
        return 'MinMaxScaler'
    elif Skew <= -1 or Skew >= 1:
        return 'RobustScaler'
    
Nan_distribution["Scaler"] = Nan_distribution["Skew"].apply(Scaler)   

In [106]:
target = "Outcome"

X = df.drop(columns=target)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    train_size=0.2,
                                                    random_state=42, 
                                                    stratify=y)

In [107]:
scaler_map ={
'StandardScaler': StandardScaler(),
'MinMaxScaler': MinMaxScaler(),
'RobustScaler': RobustScaler()
}

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

Nan_distribution['Aplicado'] = False
for column in X_train.columns:
    scaler_skew = Nan_distribution.loc[column,"Scaler"]
    scaler = scaler_map[scaler_skew]
    
    X_train_scaled[[column]] = scaler.fit_transform(X_train[[column]])
    X_test_scaled[[column]] = scaler.transform(X_test[[column]])
    Nan_distribution.loc[column, 'Aplicado'] = True
    
Nan_distribution

,Count_Nan,Percentage,Correlation,Skew,Scaler,Aplicado
Insulin,374,48.70,0.130548,2.272251,RobustScaler,True
SkinThickness,227,29.56,0.074752,0.109372,StandardScaler,True
Pregnancies,111,14.45,0.221898,0.901674,MinMaxScaler,True
BloodPressure,35,4.56,0.065068,-1.843608,RobustScaler,True
BMI,11,1.43,0.292695,-0.428982,StandardScaler,True
Glucose,5,0.65,0.466581,0.173754,StandardScaler,True
DiabetesPedigreeFunction,0,0.00,0.173844,1.919911,RobustScaler,True
Age,0,0.00,0.238356,1.129597,RobustScaler,True


In [110]:
#umbral de valores nulos
umbral = 0.1 * len(df)
#Ceros reales
real_zeros = 'Pregnancies'
#Creacion de instancia
KNNimp = KNNImputer()
# Alto numero de NAN

 

1.- EDA

In [ ]:
df["Outcome"].value_counts(normalize=True)

Outcome
0    0.651042
1    0.348958
Name: proportion, dtype: float64

In [ ]:
df_Nan['Outcome'].value_counts(normalize= True)

Outcome
0    0.651042
1    0.348958
Name: proportion, dtype: float64